In [27]:
%%html
<style>
.jp-Cell-outputArea {
    max-height: none !important;
}
</style>

In [33]:
# ==================================================
# Project Two Dashboard - Grazioso Salvare
# Built from Module Six Milestone
# ==================================================
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
import os

JupyterDash.infer_jupyter_proxy_config()

# Import your CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "*7&2XJZrCvR7"
shelter = AnimalShelter(username, password)

# Retrieve all documents from MongoDB
df_full = pd.DataFrame.from_records(shelter.read({}))

# Remove the MongoDB '_id' column to avoid DataTable crashes
if '_id' in df_full.columns:
    df_full.drop(columns=['_id'], inplace=True)

# Make sure location coordinates are numeric
df_full['location_lat'] = pd.to_numeric(df_full['location_lat'], errors='coerce')
df_full['location_long'] = pd.to_numeric(df_full['location_long'], errors='coerce')
# Age column numeric for queries
df_full['age_upon_outcome_in_weeks'] = pd.to_numeric(
    df_full['age_upon_outcome_in_weeks'], errors='coerce'
)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Encode the Grazioso Salvare logo (place the file in the same folder)
logo_filename = 'Grazioso Salvare Logo.png'
try:
    encoded_logo = base64.b64encode(open(logo_filename, 'rb').read()).decode()
except FileNotFoundError:
    encoded_logo = ''
    print("Logo file not found – dashboard will display without it.")

app.layout = html.Div([
    # ---- Branding Header ----
    html.Div(style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '20px'}, children=[
        html.A(
            html.Img(src=f'data:image/png;base64,{encoded_logo}',
                     style={'height': '80px', 'marginRight': '20px'}),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H1('SNHU CS‑340 Dashboard – Antonio Turrietta')
    ]),
    html.Div("Dashboard created by: Antonio Turrietta",
             style={'fontStyle': 'italic', 'marginBottom': '20px'}),
    html.Hr(),

    # ---- Interactive Filter Options ----
    html.Div([
        html.Label('Select Rescue Type:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'water'},
                {'label': ' Mountain / Wilderness Rescue', 'value': 'mountain'},
                {'label': ' Disaster / Individual Tracking', 'value': 'disaster'},
                {'label': ' Reset (Show All)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'block', 'margin': '5px 0'}
        )
    ], style={'marginBottom': '20px'}),
    html.Hr(),

    # ---- Interactive Data Table (enhanced from Module 6) ----
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df_full.columns],
        data=df_full.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],            # Pre‑select first row so map shows immediately
        style_table={'overflowX': 'auto'},
        style_cell={
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '300px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),
    html.Br(),
    html.Hr(),

    # ---- Charts Row: Pie Chart & Map side‑by‑side ----
    html.Div(className='row',
             style={'display': 'flex', 'justifyContent': 'space-around'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6',
                          style={'width': '48%', 'display': 'inline-block'}),
                 html.Div(id='map-id', className='col s12 m6',
                          style={'width': '48%', 'display': 'inline-block'})
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Helper: Build MongoDB query for each rescue type
def build_query(filter_type):
    if filter_type == 'water':
        return {
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'mountain':
        return {
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                              'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'disaster':
        return {
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                              'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:  # 'reset' – show all
        return {}


# Callback 1: Update data table when a filter is chosen
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'columns'),
    Input('filter-type', 'value')
)
def update_datatable(filter_type):
    query = build_query(filter_type)
    records = shelter.read(query)
    df_filtered = pd.DataFrame.from_records(records)
    if not df_filtered.empty and '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)

    # If no records match, return empty but keep all columns
    if df_filtered.empty:
        columns = [{"name": i, "id": i, "deletable": False, "selectable": True}
                   for i in df_full.columns]
        return [], columns
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True}
               for i in df_filtered.columns]
    return df_filtered.to_dict('records'), columns


# Callback 2: Highlight selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Callback 3: Pie chart updates with filtered data
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data'),
    Input('filter-type', 'value')
)
def update_pie_chart(viewData, filter_type):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available for the selected filter.")
    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    fig = px.pie(breed_counts, values='count', names='breed',
                 title=f'Breed Distribution – {filter_type.capitalize()} Rescue Candidates')
    fig.update_traces(textposition='inside', textinfo='percent+label')
    return dcc.Graph(figure=fig)


# Callback 4: Update map 
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    # Guard against empty data
    if viewData is None or len(viewData) == 0:
        return dl.Map(style={'width': '100%', 'height': '500px'},
                      center=[30.75, -97.48], zoom=10,
                      children=[dl.TileLayer()])

    dff = pd.DataFrame.from_dict(viewData)

    # Determine which row is selected (default to first)
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    if row >= len(dff):
        row = 0

    # Extract coordinates safely
    lat = dff.iloc[row]['location_lat'] if 'location_lat' in dff.columns else 30.75
    lon = dff.iloc[row]['location_long'] if 'location_long' in dff.columns else -97.48
    if pd.isna(lat) or pd.isna(lon):
        lat, lon = 30.75, -97.48   # fallback to Austin

    breed = dff.iloc[row].get('breed', 'Unknown')
    name = dff.iloc[row].get('name', 'Unknown')

    return dl.Map(style={'width': '100%', 'height': '500px'},
                  center=[lat, lon], zoom=12,
                  children=[
                      dl.TileLayer(),
                      dl.Marker(position=[lat, lon],
                                children=[
                                    dl.Tooltip(breed),
                                    dl.Popup([
                                        html.H4(name),
                                        html.P(f"Breed: {breed}")
                                    ])
                                ])
                  ])

# ==========================
# Run dashboard inline
# ==========================
app.run_server(mode='inline', height=1500)